In [2]:
# 필요 라이브러리
import pandas as pd
import warnings
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from flaml import AutoML

warnings.filterwarnings('ignore')

In [3]:
# 데이터 불러오기
df = pd.read_csv('skin_irritation_2Ddesc.csv')

y = df['label']
X = df.drop(columns=['Chemical_Name', 'standardized_smi', 'label'])

X = X.dropna(axis=1)
X = X.loc[:, X.std() >= 0.01]

print('X shape:', X.shape)
print('y 분포:')
print(y.value_counts())

X shape: (39, 144)
y 분포:
label
0    26
1    13
Name: count, dtype: int64


In [4]:
# train / test 데이터 분리
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=0,
    stratify=y
)

print('X_train shape:', X_train.shape)
print('X_test shape:', X_test.shape)

print('y_train 분포:')
print(y_train.value_counts())

print('y_test 분포:')
print(y_test.value_counts())

X_train shape: (31, 144)
X_test shape: (8, 144)
y_train 분포:
label
0    21
1    10
Name: count, dtype: int64
y_test 분포:
label
0    5
1    3
Name: count, dtype: int64


In [5]:
# train 데이터에서만 feature selection
selector = SelectKBest(f_classif, k=10)
selector.fit(X_train, y_train)

cols = X_train.columns[selector.get_support()]

X_train_sel = X_train[cols]
X_test_sel = X_test[cols]

print('선택된 descriptor:')
print(list(cols))

print('X_train_sel shape:', X_train_sel.shape)
print('X_test_sel shape:', X_test_sel.shape)

선택된 descriptor:
['MinAbsEStateIndex', 'NumValenceElectrons', 'BertzCT', 'Chi0', 'Chi1', 'SlogP_VSA6', 'HeavyAtomCount', 'MolMR', 'fr_C_O_noCOO', 'fr_ester']
X_train_sel shape: (31, 10)
X_test_sel shape: (8, 10)


In [6]:
# train 데이터에서만 FLAML + CV 실행
automl = AutoML()

automl.fit(
    X_train_sel,
    y_train,
    task='classification',
    time_budget=30,
    eval_method='cv',
    verbose=0
)

print('학습 완료!')
print('최고 모델:', automl.best_estimator)
print('최적 hyperparameter:')
print(automl.best_config)
print('train CV 정확도:', round(1 - automl.best_loss, 3))

학습 완료!
최고 모델: lgbm
최적 hyperparameter:
{'n_estimators': 12, 'num_leaves': 10, 'min_child_samples': 3, 'learning_rate': np.float64(0.37670495880280586), 'log_max_bin': 10, 'colsample_bytree': 1.0, 'reg_alpha': np.float64(0.001975258376030875), 'reg_lambda': np.float64(0.25132105398528354)}
train CV 정확도: 1.0


In [7]:
# train 데이터 예측 확인
train_pred = automl.predict(X_train_sel)

print('train 정확도:', round(accuracy_score(y_train, train_pred), 3))

train 정확도: 1.0


In [8]:
# test 데이터는 마지막 최종 평가용으로만 사용
test_pred = automl.predict(X_test_sel)

print('test 정확도:', round(accuracy_score(y_test, test_pred), 3))

print('confusion matrix:')
print(confusion_matrix(y_test, test_pred))

print('classification report:')
print(classification_report(y_test, test_pred))

test 정확도: 0.875
confusion matrix:
[[5 0]
 [1 2]]
classification report:
              precision    recall  f1-score   support

           0       0.83      1.00      0.91         5
           1       1.00      0.67      0.80         3

    accuracy                           0.88         8
   macro avg       0.92      0.83      0.85         8
weighted avg       0.90      0.88      0.87         8



In [9]:
# 30초 vs 60초 비교도 train 데이터에서만 실행
automl_long = AutoML()

automl_long.fit(
    X_train_sel,
    y_train,
    task='classification',
    time_budget=60,
    eval_method='cv',
    verbose=0
)

print('30초 최고 모델:', automl.best_estimator)
print('30초 train CV 정확도:', round(1 - automl.best_loss, 3))

print('60초 최고 모델:', automl_long.best_estimator)
print('60초 train CV 정확도:', round(1 - automl_long.best_loss, 3))

30초 최고 모델: lgbm
30초 train CV 정확도: 1.0
60초 최고 모델: xgboost
60초 train CV 정확도: 1.0


In [10]:
# k=10 vs k=20 비교도 train 데이터 기준으로 실행
selector20 = SelectKBest(f_classif, k=20)
selector20.fit(X_train, y_train)

cols20 = X_train.columns[selector20.get_support()]

X_train_sel20 = X_train[cols20]
X_test_sel20 = X_test[cols20]

automl_k20 = AutoML()

automl_k20.fit(
    X_train_sel20,
    y_train,
    task='classification',
    time_budget=30,
    eval_method='cv',
    verbose=0
)

print('========== k=10 vs k=20 비교 ==========')

print('k=10 최고 모델:', automl.best_estimator)
print('k=10 train CV 정확도:', round(1 - automl.best_loss, 3))

print('k=20 최고 모델:', automl_k20.best_estimator)
print('k=20 train CV 정확도:', round(1 - automl_k20.best_loss, 3))

========== k=10 vs k=20 비교 ==========
k=10 최고 모델: lgbm
k=10 train CV 정확도: 1.0
k=20 최고 모델: lgbm
k=20 train CV 정확도: 1.0


In [11]:
# k=20 모델도 test 데이터로 최종 평가
test_pred20 = automl_k20.predict(X_test_sel20)

print('k=20 test 정확도:', round(accuracy_score(y_test, test_pred20), 3))

print('k=20 confusion matrix:')
print(confusion_matrix(y_test, test_pred20))

k=20 test 정확도: 1.0
k=20 confusion matrix:
[[5 0]
 [0 3]]


In [12]:
# 최종 모델 저장
saved = {
    'features': list(cols),
    'model': automl
}

joblib.dump(saved, 'model_automl.joblib')

print('모델 저장 완료!')

모델 저장 완료!


In [13]:
selector.fit(X_train, y_train)
automl.fit(X_train_sel, y_train, eval_method='cv')

[flaml.automl.logger: 05-21 15:40:23] {2375} INFO - task = classification
[flaml.automl.logger: 05-21 15:40:23] {2386} INFO - Evaluation method: cv
[flaml.automl.logger: 05-21 15:40:23] {2489} INFO - Minimizing error metric: 1-roc_auc
[flaml.automl.logger: 05-21 15:40:23] {2512} WARNING - No search budget is provided via time_budget or max_iter. Training only one model per estimator. Zero-shot AutoML is used for certain tasks and estimators. To tune hyperparameters for each estimator, please provide budget either via time_budget or max_iter.
[flaml.automl.logger: 05-21 15:40:23] {2606} INFO - List of ML learners in AutoML Run: ['rf', 'lgbm', 'xgboost', 'extra_tree', 'xgb_limitdepth', 'sgd', 'lrl1']
[flaml.automl.logger: 05-21 15:40:23] {2911} INFO - iteration 0, current learner rf
[flaml.automl.logger: 05-21 15:40:25] {3046} INFO - Estimated sufficient time budget=10000s. Estimated necessary time budget=10s.
[flaml.automl.logger: 05-21 15:40:25] {3097} INFO -  at 1.5s,	estimator rf's b